# permute-back-argsort — worked example 3: Verify permute_back on all six 3D permutations

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `permute-back-argsort`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

For a 3D tensor there are exactly 6 valid permutations of `(0, 1, 2)`. The argsort-based `permute_back` must correctly invert all of them, including the identity `(0, 1, 2)` and the swap `(1, 0, 2)`. Testing all six exhaustively validates the implementation without any special-casing.

## Worked solution

**Step 1 — Enumerate all 3D permutations.** Using `itertools.permutations((0, 1, 2))` gives all 6. We test each one individually.

**Step 2 — For each permutation: forward then backward.** `y = x.permute(*dims)` applies the forward pass. `grad_x = permute_back(y, y, x, dims)` applies the backward.

**Step 3 — Check shape.** `grad_x.shape` must equal `x.shape` for every permutation.

**Step 4 — Check values.** Since permute does no arithmetic, `y.permute(*argsort(dims))` must equal `x` exactly. We assert `t.equal(grad_x, x)` — this is bit-exact equality, not approximate.

In [ ]:
import torch as t
import numpy as np
import itertools

def permute_back(grad_out: t.Tensor, out: t.Tensor, x: t.Tensor, dims: tuple) -> t.Tensor:
    inverse = tuple(int(i) for i in np.argsort(dims))
    return grad_out.permute(*inverse)

t.manual_seed(77)
x = t.randn(2, 3, 4)

all_perms = list(itertools.permutations((0, 1, 2)))
print(f'Testing {len(all_perms)} permutations...')

for dims in all_perms:
    y = x.permute(*dims)
    grad_x = permute_back(y, y, x, dims)
    shape_ok = grad_x.shape == x.shape
    value_ok = t.equal(grad_x, x)
    status = 'PASS' if (shape_ok and value_ok) else 'FAIL'
    print(f'dims={dims}: {status}  grad_x.shape={grad_x.shape}')

print('All round-trips verified.')